### [ 주제: 시과& 바나나 분류- (1) 데이터 수집 ]
- **[1] 웹 크롤링**
- **[2] ROI 추출 및 저장**

- **데이터 저장 경로**
''' text
    Data
        ├─ Images
        │  ├─ apple
        │  └─ banana
        └─ Roi
           ├─ apple
           └─ banana

           '''

In [2]:
# %pip install icrawer
# %pip install beautifulsoup4
# %pip install --upgrade beautifulsoup4

**[0] 사전준비**
- 모듈로딩
- 폴더 체크 및 생성

In [7]:
## --------------------------------------------------
## 모듈 로딩
## --------------------------------------------------
from icrawler.builtin import BingImageCrawler  ## 웹 크롤링
import os                                      ## 경로, 파일 관련
import cv2


## --------------------------------------------------
## 전역 변수 및 상수
## --------------------------------------------------
## => 이미지 관련
IMG_DIR = '../Data/Images'
ROI_DIR  = '../Data/Roi'

## => 수집 데이터
DATA_LABEL= ['apple', 'banana']

APPLE_DIR= f'{IMG_DIR}/apple'
APPLE_ROI= f'{ROI_DIR}/apple'


BANANA_DIR= f'{IMG_DIR}/banana'
BANANA_ROI= f'{ROI_DIR}/banana'


# 경로 확인
print(IMG_DIR, APPLE_DIR, BANANA_DIR, sep='\n')
print(ROI_DIR, APPLE_ROI, BANANA_ROI, sep='\n')

../Data/Images
../Data/Images/apple
../Data/Images/banana
../Data/Roi
../Data/Roi/apple
../Data/Roi/banana


In [8]:
## 폴더 생성 및 체크
## --------------------------------------------------
os.makedirs(APPLE_DIR, exist_ok=True)
os.makedirs(APPLE_ROI, exist_ok=True)
os.makedirs(BANANA_DIR, exist_ok=True)
os.makedirs(BANANA_ROI, exist_ok=True)

In [9]:
## - 저장 위치 : ../Data/Images/수집과일명
## --------------------------------------------------
## => 폴더명, 검색어
DIR_KEY = [['apple', '사과'], ['banana', '바나나']]

## => 필터 조건 설정
google_filters = {
    'type': 'photo',       # 만화, 일러스트, 텍스트를 제외한 '실제 사진'만 선택
    'color': 'color',      # 흑백 이미지 제외 (컬러 사진만)
    'size': 'large'        # 해상도가 높은 큰 이미지 위주로 선택
}

## => 검색 조건에 따른 웹 크롤링 후 저장
for dirname, keyword in DIR_KEY:
    ## => 저장 폴더 생성
    CROL_DIR = f'{IMG_DIR}/{dirname}'
    os.makedirs(IMG_DIR, exist_ok=True)

    ## => 저장할 디렉토리 지정 및 크롤러 생성
    crawler = BingImageCrawler(storage={'root_dir': CROL_DIR})

    crawler.crawl(keyword=keyword, filters=google_filters, max_num=100)

2026-06-17 13:45:02,264 - INFO - icrawler.crawler - start crawling...
2026-06-17 13:45:02,264 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-06-17 13:45:02,265 - INFO - feeder - thread feeder-001 exit
2026-06-17 13:45:02,267 - INFO - icrawler.crawler - starting 1 parser threads...
2026-06-17 13:45:02,268 - INFO - icrawler.crawler - starting 1 downloader threads...
2026-06-17 13:45:04,013 - INFO - parser - parsing result page https://www.bing.com/images/async?q=사과&first=0&qft=+filterui:photo-photo+filterui:color2-color+filterui:imagesize-large
2026-06-17 13:45:05,718 - INFO - downloader - image #1	https://recipe1.ezmember.co.kr/cache/recipe/2021/10/28/daed301699a29ae598f71080517f976e1.jpg
2026-06-17 13:45:08,082 - INFO - downloader - image #2	https://happyyumblog.co.kr/wp-content/uploads/2024/12/제목을-입력해주세요_-001-2-1-optimized.jpg
2026-06-17 13:45:09,871 - INFO - downloader - image #3	https://health.chosun.com/site/data/img_dir/2023/05/24/2023052402062_0.jpg
2026-06-17 13:4

In [ ]:
## --------------------------------------------------
## 사과, 배, 바나나 크롤링 이미지에서 ROI 추출 후 저장
## - 이미지 위치 : ../Data/Images/과일명폴더
## - 저장 위치 : ../Data/Roi/수집과일명폴더/파일명
## --------------------------------------------------
## => 과일명 폴더 내 이미지 파일 추출
for dname, sname in [[APPLE_DIR,APPLE_ROI],  [BANANA_DIR, BANANA_ROI]]:

    ## 이미지 리스트 추출
    filelist = os.listdir(dname)
    print(f'filelist : {len(filelist)}개')

    for filename in filelist:
        ## 저장 파일 경로 + 파일명
        SAVE_PATH = f'{dname}/{filename}'
        FILE_PATH = f'{dname}/{filename}'

        ## ROI 영역 추출
        imgNP = cv2.imread(FILE_PATH)
        x, y, w, h = cv2.selectROI("SELECT", imgNP)

        ## ROI 영역 파일 저장
        roi = imgNP[y:y+h, x:x+w]
        cv2.imwrite(SAVE_PATH, roi)


        ## 키처리
        cv2.waitKey()
        cv2.destroyAllWindows()

        break
    break

filelist : 45개


### [ 주제 : 사과 & 바나나 분류 - (2) 데이터 전처리 ]
- **[1] 전처리 방법 결정**
- **[2] 훈련용 데이터셋 생성**

- **데이터셋 구조**
  ```text
  이미지의 행×열 픽셀값 라벨
  이미지의 행×열 픽셀값
  이미지의 행×열 픽셀값
  이미지의 행×열 픽셀값
  이미지의 행×열 픽셀값
  이미지의 행×열 픽셀값
  이미지의 행×열 픽셀값
'''

- **전처리 방법**
  - 크기일치
  - 흑백이미지로 해서 모양만으로도 판별가능하게 